# Component_01 — STAGE 1: Report Target Cleaning
### Eliminating prior-study hallucination from MIMIC-CXR training targets

---

## What this notebook does

Your report generator hallucinates references to previous X-rays it never saw
(*"As compared to the previous radiograph, there is no relevant change..."*).

**This is not a model bug.** It is a faithful reproduction of the training data:

| Un-groundable language in your 46,274 training targets | % of reports |
|---|---|
| "unchanged / no change / stable / constant" | 47.5% |
| `___` de-identification placeholder | 27.6% |
| "prior / previous / earlier study" | 21.3% |
| "as compared to / in comparison with" | 19.6% |
| "again seen / persistent / remains" | 19.3% |
| "interval change / new since" | 14.9% |
| **ANY prior-reference language** | **69.7%** |
| **Sentences impossible to ground in one image** | **24.7%** |

Seven out of ten examples taught the model that a correct report mentions a prior
study. It has never seen one, so it invents one — and it *amplifies*: 63% of
generated reports contain prior-language versus 51% of references.

**This notebook rebuilds every target so that every sentence is groundable in the
single image in front of it.**

## What it produces

```
clean_train.csv / clean_val.csv / clean_test.csv   <- new training targets
stage1_audit.json                                  <- full before/after metrics
stage1_examples.txt                                <- 40 before/after pairs to eyeball
```

## Success criteria (the notebook checks these itself)

- Residual prior-reference rate **< 2%** (from 69.7%)
- Corpus retention **> 92%**
- Zero rows with empty targets
- Constant-baseline recomputed on the cleaned corpus

---

## ⚠️ GPU: NOT NEEDED FOR THIS NOTEBOOK

**Stage 1 is 100% CPU. Select `Runtime -> Change runtime type -> CPU`.**

A CPU runtime on Colab costs **zero compute units**. If you run this on a GPU
runtime you will burn units for nothing. GPU guidance for Stage 4 is at the bottom
of this notebook.

---
# 0 · Environment

In [ ]:
# ── Verify we are on CPU (this notebook must not burn compute units) ──────────
import subprocess, sys, os, platform

print("=" * 78)
print("  COMPONENT_01 · STAGE 1 · ENVIRONMENT CHECK")
print("=" * 78)

try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10)
    gpu = out.stdout.strip()
except Exception:
    gpu = ""

if gpu:
    print(f"  ⚠️  GPU RUNTIME DETECTED: {gpu}")
    print("     This notebook is CPU-only. A GPU runtime burns compute units for nothing.")
    print("     Runtime -> Change runtime type -> CPU, then re-run.")
else:
    print("  ✅ CPU runtime — 0 compute units will be consumed.")

print(f"  Python {platform.python_version()}  |  {platform.platform()}")
print("=" * 78)

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
# rouge-score is only needed for the constant-baseline control at the end.
import importlib

need = []
for mod, pkg in [("rouge_score", "rouge-score"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(mod) is None:
        need.append(pkg)

if need:
    print(f"Installing: {' '.join(need)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Done.")
else:
    print("All dependencies present.")

import re, json, unicodedata, random
from pathlib import Path
from collections import Counter, OrderedDict
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
RNG_SEED = 42
random.seed(RNG_SEED); np.random.seed(RNG_SEED)
print(f"pandas {pd.__version__} | numpy {np.__version__}")

---
# 1 · Mount Drive & configure paths

**Before running this cell**, upload these files to Google Drive at
`MyDrive/Component_01/data/raw/`:

| File | Size | Required |
|---|---|---|
| `cardio_train.csv` | 36 MB | ✅ yes |
| `cardio_val.csv` | 4.5 MB | ✅ yes |
| `cardio_test.csv` | 4.7 MB | ✅ yes |
| `mimic-cxr-2.0.0-chexpert.csv` | 9.3 MB | optional (official-label cross-check) |

Upload the folder through the Drive web UI — it is far faster than uploading
through Colab. Total is ~55 MB, about a minute.

In [ ]:
# ── Mount Drive ──────────────────────────────────────────────────────────────
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/Component_01")
else:
    # Local fallback — point this at your project root
    PROJECT = Path("./Component_01")

RAW_DIR    = PROJECT / "data" / "raw"
CLEAN_DIR  = PROJECT / "data" / "stage1_clean"
REPORT_DIR = PROJECT / "reports" / "stage1"

for d in (RAW_DIR, CLEAN_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT   :", PROJECT)
print("RAW_DIR   :", RAW_DIR)
print("CLEAN_DIR :", CLEAN_DIR)
print("REPORT_DIR:", REPORT_DIR)

In [ ]:
# ── Verify inputs are present before doing anything ──────────────────────────
SPLITS = ["train", "val", "test"]
SRC = {s: RAW_DIR / f"cardio_{s}.csv" for s in SPLITS}
CHEXPERT_CSV = RAW_DIR / "mimic-cxr-2.0.0-chexpert.csv"

print("=" * 78)
print("  INPUT FILE CHECK")
print("=" * 78)
missing = []
for s, p in SRC.items():
    if p.exists():
        print(f"  ✅ {p.name:<20} {p.stat().st_size/1e6:>7.1f} MB")
    else:
        print(f"  ❌ {p.name:<20} MISSING")
        missing.append(p)

if CHEXPERT_CSV.exists():
    print(f"  ✅ {CHEXPERT_CSV.name:<20} {CHEXPERT_CSV.stat().st_size/1e6:>7.1f} MB  (optional)")
    HAVE_CHEXPERT = True
else:
    print(f"  ⚠️  {CHEXPERT_CSV.name:<20} not found — official-label cross-check will be skipped")
    HAVE_CHEXPERT = False

if missing:
    raise FileNotFoundError(
        "Upload the missing CSV(s) to " + str(RAW_DIR) + " then re-run this cell.")
print("=" * 78)
print("  All required inputs present.")

---
# 2 · Load data & integrity audit

Before changing anything, verify the dataset is what we think it is.

In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
DF = {}
for s in SPLITS:
    DF[s] = pd.read_csv(SRC[s], low_memory=False)
    print(f"  {s:<6} {len(DF[s]):>7,} rows  x  {DF[s].shape[1]} cols")

ALL = pd.concat([DF[s].assign(_split=s) for s in SPLITS], ignore_index=True)
print(f"  {'TOTAL':<6} {len(ALL):>7,} rows")

LABEL_COLS = ["Cardiomegaly", "Edema", "Pleural_Effusion", "Atelectasis",
              "Consolidation", "Lung_Opacity", "Pneumonia", "Pneumothorax"]
if "No_Finding" in ALL.columns:
    LABEL_COLS_FULL = LABEL_COLS + ["No_Finding"]
else:
    LABEL_COLS_FULL = LABEL_COLS

required = {"dicom_id", "subject_id", "study_id", "impression_text", "findings_text"}
missing_cols = required - set(ALL.columns)
if missing_cols:
    raise KeyError(f"Expected columns missing from source CSVs: {missing_cols}")
print("\n  ✅ All required columns present.")

In [ ]:
# ── Integrity audit ──────────────────────────────────────────────────────────
print("=" * 78)
print("  DATASET INTEGRITY AUDIT")
print("=" * 78)

integrity = OrderedDict()

# Split sizes
for s in SPLITS:
    integrity[f"n_{s}"] = int(len(DF[s]))
integrity["n_total"] = int(len(ALL))

# Leakage — the single most important check
sub = {s: set(DF[s]["subject_id"]) for s in SPLITS}
std = {s: set(DF[s]["study_id"])   for s in SPLITS}
leaks = {
    "patient_train_val":  len(sub["train"] & sub["val"]),
    "patient_train_test": len(sub["train"] & sub["test"]),
    "patient_val_test":   len(sub["val"]   & sub["test"]),
    "study_train_val":    len(std["train"] & std["val"]),
    "study_train_test":   len(std["train"] & std["test"]),
    "study_val_test":     len(std["val"]   & std["test"]),
}
integrity["leakage"] = leaks
print("\n  LEAKAGE (all must be 0):")
for k, v in leaks.items():
    print(f"    {k:<22} {v:>6}  {'✅' if v == 0 else '❌ LEAK'}")
if any(leaks.values()):
    print("\n  ❌ LEAKAGE DETECTED — stop and fix the split before training.")
else:
    print("\n  ✅ Zero patient and study leakage.")

# Duplicates
dup_dicom = int(ALL["dicom_id"].duplicated().sum())
integrity["duplicate_dicom_ids"] = dup_dicom
print(f"\n  Duplicate dicom_ids: {dup_dicom}  {'✅' if dup_dicom == 0 else '❌'}")

# View distribution
if "ViewPosition" in ALL.columns:
    vc = ALL["ViewPosition"].value_counts().to_dict()
    integrity["view_position"] = {str(k): int(v) for k, v in vc.items()}
    print(f"  ViewPosition: {vc}")

# Label prevalence
print(f"\n  {'Label':<20}{'train pos':>12}{'prev%':>9}")
prev = {}
for c in LABEL_COLS_FULL:
    if c in DF["train"].columns:
        pos = int((DF["train"][c] == 1).sum())
        prev[c] = {"pos": pos, "prev_pct": round(pos / len(DF["train"]) * 100, 2)}
        print(f"  {c:<20}{pos:>12,}{prev[c]['prev_pct']:>9.2f}")
integrity["train_label_prevalence"] = prev

# Co-pathology richness
pc = [c for c in LABEL_COLS if c in DF["train"].columns]
cnt = (DF["train"][pc] == 1).sum(axis=1)
integrity["mean_positive_labels_per_image"] = round(float(cnt.mean()), 3)
print(f"\n  Mean positive labels/image (train): {cnt.mean():.2f}")
print(f"  Distribution: {cnt.value_counts().sort_index().to_dict()}")
print("=" * 78)

---
# 3 · Pre-clean audit — measure the hallucination source

This reproduces the 69.7% figure from your own data so you have it verified
inside the notebook, and gives us the baseline to measure the fix against.

In [ ]:
# ── Detection patterns (also used as the post-clean QA gate) ─────────────────
DETECT = OrderedDict([
    ("comparison_opener",
     r"(as compared (to|with)|in compar(ison|ed) (to|with)|compared (to|with) (the )?(prior|previous|earlier|exam|study|radiograph))"),
    ("prior_study_ref",
     r"\b(prior|previous|earlier|preceding)\s+(study|studies|exam|examination|radiograph|film|chest|imaging|ct)"),
    ("stability_claim",
     r"\b(unchanged|no (significant |relevant |interval )?change|no interval change|stable|constant)\b"),
    ("temporal_change",
     r"\b(interval(ly)? (increase|decrease|worsen|improv|develop|resolution|removal)|has (improved|worsened|resolved|increased|decreased)|new(ly)? (from|since)|since (the )?(prior|previous|___))"),
    ("deident_placeholder", r"___"),
    ("persistence_claim",
     r"\b(again (seen|noted|demonstrated|identified)|persistent(ly)?|persists|remains?)\b"),
    # v3: comparative CHANGE language. Implies a previous image just as strongly as
    # "prior study" does, but contains none of those words — the v1/v2 detector was
    # blind to it and 11.25% of cleaned rows still carried it.
    ("comparative_change",
     r"\b(increasing|decreasing|worsening|improving|progressing"
     r"|(has|have) been (an? )?(increase|decrease|improvement|worsening|progression)"
     r"|(increase|decrease|reduction) (in|of)"
     r"|(is|are|was|were|appears?) similar"
     r"|re-?accumulat|re-?expansion)\b"),
])
DETECT_ANY = re.compile("|".join(f"(?:{p})" for p in DETECT.values()), re.I)


def audit_prior_language(series, label):
    txt = series.fillna("").astype(str).str.lower()
    n = len(txt)
    res = OrderedDict()
    print(f"\n  {label}  (n={n:,})")
    print(f"  {'-' * 62}")
    for name, pat in DETECT.items():
        hits = int(txt.str.contains(pat, regex=True, na=False).sum())
        res[name] = {"n": hits, "pct": round(hits / n * 100, 2)}
        print(f"    {name:<24}{hits:>9,}  {hits/n*100:>6.2f}%")
    any_hits = int(txt.str.contains(DETECT_ANY.pattern, regex=True, na=False).sum())
    res["ANY"] = {"n": any_hits, "pct": round(any_hits / n * 100, 2)}
    print(f"  {'-' * 62}")
    print(f"    {'>>> ANY prior-reference':<24}{any_hits:>9,}  {any_hits/n*100:>6.2f}%")
    return res


def build_original_target(row):
    # Reproduce exactly how the current pipeline builds report_text:
    # impression + findings (impression FIRST — inverted vs real radiology order)
    imp = str(row.get("impression_text", "") or "").strip()
    fin = str(row.get("findings_text", "") or "").strip()
    imp = "" if imp.lower() in ("nan", "none") else imp
    fin = "" if fin.lower() in ("nan", "none") else fin
    return (imp + " " + fin).strip()


print("=" * 78)
print("  PRE-CLEAN AUDIT — un-groundable language in current targets")
print("=" * 78)

ALL["_orig_target"] = ALL.apply(build_original_target, axis=1)
pre_audit = audit_prior_language(ALL["_orig_target"], "ALL SPLITS · current targets")

In [ ]:
# ── Sentence-level rate + length + format consistency ────────────────────────
def split_sentences(text):
    # Radiology-aware sentence splitter.
    if not text:
        return []
    t = re.sub(r"\s+", " ", str(text)).strip()
    if not t:
        return []
    # Protect abbreviations and numbered-list markers from the splitter
    prot = [("Dr.", "Dr<D>"), ("Mr.", "Mr<D>"), ("Mrs.", "Mrs<D>"), ("Ms.", "Ms<D>"),
            ("vs.", "vs<D>"), ("e.g.", "e<D>g<D>"), ("i.e.", "i<D>e<D>"),
            ("approx.", "approx<D>"), ("cf.", "cf<D>"), ("No.", "No<D>")]
    for a, b in prot:
        t = t.replace(a, b)
    # Protect "1. " / "2. " numbered-list markers from the sentence splitter.
    # The (?<![:\d]) guard is essential: without it a clock time like "13:45. The
    # known pneumothorax..." has its "45." protected, so the splitter never breaks
    # there and two sentences are merged — letting one sentence's fate delete the
    # other's findings.
    t = re.sub(r"(?<![:\d])\b(\d{1,2})\.\s+", r"\1<D> ", t)
    parts = re.split(r"(?<=[.;!?])\s+", t)
    out = []
    for p in parts:
        p = p.replace("<D>", ".").strip()
        if p:
            out.append(p)
    return out


samp = ALL["_orig_target"].sample(min(6000, len(ALL)), random_state=RNG_SEED)
tot_s = bad_s = 0
for t in samp:
    for s in split_sentences(t):
        if len(s) < 5:
            continue
        tot_s += 1
        if DETECT_ANY.search(s):
            bad_s += 1
sent_rate = bad_s / max(tot_s, 1) * 100
print(f"\n  SENTENCE-LEVEL ({len(samp):,} report sample):")
print(f"    {bad_s:,} / {tot_s:,} = {sent_rate:.1f}% of sentences are un-groundable")

wl = ALL["_orig_target"].str.split().str.len()
print(f"\n  TARGET LENGTH (words): mean={wl.mean():.0f} median={wl.median():.0f} "
      f"p90={wl.quantile(.9):.0f} p99={wl.quantile(.99):.0f} max={wl.max()}")

imp_ok = ALL["impression_text"].fillna("").astype(str).str.strip().str.len() > 5
fin_ok = ALL["findings_text"].fillna("").astype(str).str.strip().str.len() > 5
print(f"\n  SECTION AVAILABILITY:")
print(f"    both sections   : {(imp_ok & fin_ok).mean()*100:>5.1f}%")
print(f"    impression only : {(imp_ok & ~fin_ok).mean()*100:>5.1f}%")
print(f"    findings only   : {(~imp_ok & fin_ok).mean()*100:>5.1f}%")
print(f"    neither         : {(~imp_ok & ~fin_ok).mean()*100:>5.1f}%")
print("\n  ⚠️  Inconsistent target format is a second driver of template collapse:")
print("     the model cannot tell which format to emit, so it averages into a template.")

pre_stats = {
    "sentence_level_ungroundable_pct": round(sent_rate, 2),
    "target_len_mean": round(float(wl.mean()), 1),
    "target_len_median": float(wl.median()),
    "target_len_max": int(wl.max()),
    "sections_both_pct": round(float((imp_ok & fin_ok).mean() * 100), 1),
    "sections_impression_only_pct": round(float((imp_ok & ~fin_ok).mean() * 100), 1),
    "sections_findings_only_pct": round(float((~imp_ok & fin_ok).mean() * 100), 1),
}

---
# 4 · The cleaning engine

### Design

Naive deletion of every sentence containing "unchanged" or "stable" would destroy
real findings — *"Unchanged moderate cardiomegaly with bilateral effusions"* carries
two genuine pathologies. So the engine is **content-gated**, not keyword-blind:

| Step | Action |
|---|---|
| **A** | Strip leading comparison clauses (*"As compared to the prior study, ..."*) |
| **B** | Strip temporal/comparative modifiers in place (*"unchanged X"* → *"X"*) |
| **C** | Remove `___` placeholders and repair punctuation |
| **D** | **Content gate** — drop any sentence left with no anatomical/pathological term |
| **E** | Drop communication & recommendation sentences (not image-groundable) |
| **F** | Rebuild as `FINDINGS: ... IMPRESSION: ...` in the *correct* radiological order |
| **G** | Drop rows under 8 words |

The content gate is what makes this safe: *"there is no relevant change"* contains no
clinical term and is dropped, while *"moderate cardiomegaly"* is kept.

In [ ]:
# ── Step E: sentences to drop outright (communication / recommendation) ──────
HARD_DROP = re.compile(
    r"("
    r"was (paged|notified|called|contacted)|were (paged|notified|called|contacted)"
    r"|(findings|results) (were|was) (reported|discussed|communicated)"
    r"|discussed (with|by) (dr|the)"
    r"|by telephone|via telephone|by phone|telephone at"
    r"|recommend(ed|ation)?\b"
    r"|please correlate|clinical correlation is recommended"
    r"|wet read|attending radiologist"
    # REMOVED: "comparison (is|was)? (made)? (with|to)".
    # It was intended for contentless sentences like "Comparison is made with the
    # prior study." but it also destroyed sentences carrying REAL findings, e.g.
    # "In comparison with the study of ___, there has been development of a
    # moderate post-procedure pneumothorax." -> whole sentence deleted, and with it
    # the only mention of the pneumothorax. Cost 4.1% of pneumothorax positives,
    # 2.6% of edema, 2.2% of pneumonia.
    # Genuinely contentless comparison sentences are already removed by
    # LEAD_CLAUSE + RESIDUAL_PRIOR + the CONTENT gate, so nothing is lost.
    r"|no (previous|prior) (study|studies|exam|imaging) (is |are )?available"
    r")", re.I)

# ── Step A: leading comparison clauses to strip ─────────────────────────────
LEAD_CLAUSE = [
    re.compile(r"^\s*as compared (to|with)[^,.;]*[,.;]\s*", re.I),
    re.compile(r"^\s*in comparison (with|to)[^,.;]*[,.;]\s*", re.I),
    re.compile(r"^\s*compared (to|with)[^,.;]*[,.;]\s*", re.I),
    re.compile(r"^\s*(in )?the interval[^,.;]*[,.;]\s*", re.I),
    re.compile(r"^\s*since (the )?[^,.;]*[,.;]\s*", re.I),
    re.compile(r"^\s*relative to[^,.;]*[,.;]\s*", re.I),
]

# ── Step B: temporal / comparative modifiers stripped in place ──────────────
# Order matters — longer phrases first.
#
# NOTE ON REGEX SHAPE: patterns must NOT end with `?\b` after an optional group.
# `\bunchanged (in )?(position)?\b` FAILS on "unchanged." because after consuming
# "unchanged " the next char is "." and there is no word boundary between a space
# and a period. Trailing optional groups are therefore written WITHOUT a closing \b.
# This bug leaked 19% of stability claims through the first build — do not reintroduce it.
#
PHRASE_STRIP = [
    # --- explicit comparison phrases (most specific first) -------------------
    # Nouns MUST carry `s?` — "compared to the exams" otherwise matches only
    # "...exam" and strands the orphan letter "s". Third occurrence of the
    # plural-boundary bug; check every noun alternation for it.
    r"\bas compared (to|with)( the)?( prior| previous| earlier)?( radiographs?| studies| study| exams?| examinations?| films?)?",
    r"\bin comparison (with|to)( the)?( prior| previous| earlier)?( radiographs?| studies| study| exams?| examinations?| films?)?",
    r"\bcompared (to|with)( the)?( prior| previous| earlier| most recent)?( radiographs?| studies| study| exams?| examinations?| films?)?",
    r"\b(when )?compared (to|with)\b",
    # Trailing noun must be consumed, else "similar to prior studies" leaves "studies".
    r"\bsimilar (to|in appearance to)( the)?( prior| previous| earlier)"
    r"( studies| study| exams?| examinations?| radiographs?| films?| imaging| images?)?",
    r"\b(is|are|was|were|appears?|appeared)( now)? similar\b",
    r"\band similar\b",
    r"\brelative to( the)?( prior| previous| earlier)( exam| examination| study| radiograph| film)?",
    r"\bsince( the)?( prior| previous| earlier)( study| exam| radiograph)?",
    r"\bfrom( the)?( prior| previous| earlier)( study| exam| radiograph)?",
    # Prepositional references to prior imaging, INCLUDING PLURALS.
    # Strips the reference but keeps the finding: "Mild opacity is stable across
    # multiple prior radiographs." -> "Mild opacity."  Without this the whole
    # sentence would be dropped by RESIDUAL_PRIOR and the finding lost.
    r"\b(as (seen|noted|demonstrated|described) (on|in) |compared (to|with) |across |over |from |on |in |than |versus |vs )?"
    r"(multiple |several |all |the |any |numerous )*"
    r"(prior|previous|earlier|preceding)\s+"
    r"(studies|study|exams?|examinations?|radiographs?|films?|imaging|images?|chest|cts?)\b",
    # --- interval / temporal -------------------------------------------------
    r"\bin the interval\b", r"\binterval(ly)?\b",
    r"\bpreviously\b( seen| noted| demonstrated| identified| described| imaged| visualized)?",
    r"\bre-?demonstrated\b",
    # \b after "again" is MANDATORY: without it this matches inside "against"
    # ("lies against the chest wall") and leaves the orphan fragment "st".
    r"\bagain\b( seen| noted| demonstrated| identified| visualized| present)?",
    # --- stability / no-change ----------------------------------------------
    # Trailing preposition is consumed too, otherwise "no significant change in
    # the extent of X" leaves the stub "There is in the extent of X".
    r"\bno (significant |relevant |appreciable |substantial )?(interval )?changes?\b( is| are)?( seen| noted| identified| demonstrated)?( in| to| from| of| with)?",
    r"\b(little|no)( overall| significant| relevant)? changes?\b( in| to| from| of)?",
    # "<verb> unchanged in <anything>" must be consumed WHOLE, else
    # "clips are unchanged in location" leaves the stub "clips are location".
    r"\b(is|are|was|were)( now)? (unchanged|stable) in( its)? \w+",
    r"\b(unchanged|stable) in( its)? \w+",
    r"\b(is|are|was|were)( now)? (unchanged|stable)\b",
    r"\bunchanged\b( in)?( position| appearance| size| extent| severity| location| configuration| alignment| course| caliber| contour| distribution)?",
    r"\bstable\b( in)?( appearance| position| size| location| configuration| alignment)?",
    r"\bstability\b",
    r"\bpersistent(ly)?\b", r"\bpersists?\b", r"\bpersisting\b",
    r"\bcontinues? to be\b", r"\bcontinued\b",
    # \b MUST sit immediately after remains? — otherwise "remainder" matches
    # "remain" and is corrupted into "der". This hit 80 real reports.
    r"\bremains?\b( unchanged| stable| similar)?",
    r"\bconstant\b",
    # --- novelty / progression ----------------------------------------------
    r"\bnewly\b( developed| seen| noted| identified| appeared| occurred| present)?",
    r"\bnew (from|since)\b",
    r"\bnew\b",
    # v3: comparative CHANGE language — implies a prior image without ever naming one.
    # CRITICAL: every terminal alternation needs \b IMMEDIATELY after it.
    # Without it "increase" matches inside "increased" and leaves the orphan
    # letter "d" — "the pleural effusion has increased." -> "The pleural effusion d."
    # Same failure class as remainder->der. This hit 2.5% of rows in v3.
    r"\b(there )?(has|have)( been)?( a| an)?( interval| slight| mild| marked| significant| further)? "
    r"(increase|decrease|improvement|worsening|progression|development|resolution|reduction)\b( in| of)?",
    r"\b(has|have)( been)? (improved|worsened|resolved|increased|decreased|cleared|progressed|developed)\b",
    r"\b(an? )?(interval |slight |mild |marked |significant |further |continued |overall |gradual )?"
    r"(increase|decrease|reduction)\b( in| of)( the)?( size| extent| degree| severity| amount| number)?( of)?",
    # Bare change nouns: "Improvement in left pleural effusion."
    r"\b(marked |significant |slight |mild |overall |interval |continued )?"
    r"(improvement|worsening|progression|resolution|clearing)\b( in| of)",
    r"\b(slightly |mildly |markedly |significantly |minimally |progressively |gradually )?"
    r"(increasing|decreasing|improving|worsening|progressing)\b",
    r"\b(slightly |mildly |markedly |significantly |minimally )?(increased|decreased|improved|worsened|progressed)\b",
    r"\bresolution of\b", r"\bresolved\b",
    r"\bprogression of\b",
    r"\bno longer\b( seen| visualized| identified| present)?",
    r"\bstatus post (interval )?(removal|placement) of\b",
    r"\bpre-?existing\b", r"\bongoing\b", r"\bstill\b", r"\blongstanding\b",
    # --- catch-all, last ------------------------------------------------------
    r"\bsince\b",
    # Bare "prior/previous/earlier" used adjectivally: "Prior opacity at the
    # left base." Still asserts a previous image. Runs last so the specific
    # multi-word patterns above get first refusal.
    r"\b(prior|previous|earlier|preceding)\b",
]
PHRASE_STRIP_RE = [re.compile(p, re.I) for p in PHRASE_STRIP]

# Rewrites applied BEFORE the strip loop. The "re-" prefix asserts the finding was
# there before and went away — un-groundable. The base noun is still true of THIS
# image, so rewrite rather than delete.
PHRASE_REWRITE = [
    (re.compile(r"\bre-?accumulation\b", re.I), "accumulation"),
    (re.compile(r"\bre-?accumulated\b", re.I), "present"),
    (re.compile(r"\bre-?expansion\b", re.I), "expansion"),
    (re.compile(r"\bre-?expanded\b", re.I), "expanded"),
]

# Sentences that reference a prior study but survived the above -> drop
RESIDUAL_PRIOR = re.compile(
    r"\b(prior|previous|earlier|preceding)\s+"
    r"(studies|study|exams?|examinations?|radiographs?|films?|chest|imaging|images?|cts?|comparisons?)\b",
    re.I)

# ── Step D: clinical content vocabulary (the safety gate) ────────────────────
CONTENT_TERMS = [
    # anatomy
    "lung", "lungs", "pulmonary", "heart", "cardiac", "cardiomediastinal", "mediastin",
    "hilar", "hila", "hilum", "pleura", "pleural", "diaphragm", "costophrenic", "aorta",
    "aortic", "thorac", "chest", "rib", "ribs", "spine", "spinal", "vertebra", "clavicle",
    "sternot", "sternum", "apex", "apical", "base", "basilar", "basal", "lobe", "lobar",
    "airway", "trachea", "bronch", "carina", "azygos", "svc", "vascular", "vessel",
    "silhouette", "contour", "soft tissue", "osseous", "bony", "bone", "abdomen",
    # pathology
    "cardiomegaly", "enlarge", "effusion", "pneumothorax", "edema", "consolidation",
    "opacity", "opacities", "opacification", "atelecta", "pneumonia", "infiltrate",
    "nodule", "mass", "lesion", "fracture", "emphysema", "fibrosis", "scarring",
    "granuloma", "calcif", "congestion", "interstitial", "airspace", "air space",
    "hyperinflat", "blunting", "thickening", "collapse", "infection", "aspiration",
    "hemothorax", "pneumoperitoneum", "free air", "hernia", "kyphosis", "scoliosis",
    "degenerative", "volume", "volumes", "aeration", "perihilar", "retrocardiac",
    "septal", "kerley", "reticular", "nodular", "patchy", "focal", "diffuse",
    "cavitation", "bulla", "cyst", "elevation", "deviation", "widening", "prominence",
    # devices (visible in the image -> groundable)
    "catheter", "tube", "line", "picc", "pacemaker", "pacer", "icd", "wire", "wires",
    "clip", "clips", "stent", "valve", "device", "port", "drain", "sheath", "electrode",
    "swan-ganz", "endotracheal", "nasogastric", "feeding", "tracheostomy", "defibrillator",
    # normality assertions (groundable and clinically meaningful)
    "clear", "normal", "unremarkable", "within normal limits", "no acute", "intact",
]
CONTENT_RE = re.compile("|".join(re.escape(t) for t in CONTENT_TERMS), re.I)


# Function words left dangling at the end of a sentence after modifier removal
# ("...support devices are." / "...cardiomegaly is." / "...edema since.")
DANGLING = re.compile(
    r"\b(is|are|was|were|and|or|of|the|a|an|with|without|to|from|in|on|at|for|by"
    r"|since|than|as|but|that|which|has|have|had|been|be|also|now|then|when"
    # adverbs stranded when their verb was removed:
    # "The left pleural effusion has improved significantly." -> "... significantly."
    r"|significantly|essentially|slightly|markedly|minimally|mildly|moderately"
    r"|further|overall|again|likely|probably|possibly)"
    # Trailing punctuation MUST be optional: stripping a modifier usually consumes
    # the sentence's own period too, so "...views of the chest are" arrives here
    # with nothing after it. With [.,;:] mandatory this never fired and the
    # stranded copula survived into the corpus.
    r"\s*[.,;:]?\s*$", re.I)

# Function words left stranded at the START of a sentence after modifier removal
# ("Again seen is a small effusion." -> "Is a small effusion." -> "A small effusion.")
# Deliberately does NOT strip leading the/a/an — those are grammatical.
LEADING = re.compile(
    r"^(is|are|was|were|and|or|but|then|also|of|to|from|with)\b\s*", re.I)


_WS = re.compile(r"\s{2,}")


def _punct_fix(s):
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([,.;:])", r"\1", s)
    s = re.sub(r"([,;:])\1+", r"\1", s)
    s = re.sub(r"[,;:]\s*\.", ".", s)
    s = re.sub(r"\.{2,}", ".", s)
    s = re.sub(r"^[\s,;:.\-]+", "", s)
    s = re.sub(r"\s{2,}", " ", s)
    return s.strip()


def _tidy(s):
    # Repair whitespace and punctuation after aggressive substitution.
    s = _punct_fix(s)
    # Collapse artefacts like "there is ." / "there are ,"
    s = re.sub(r"\bthere (is|are)\s*[.,;:]", "", s, flags=re.I)
    s = re.sub(r"\bis\s+(is|are)\b", r"\1", s, flags=re.I)
    s = re.sub(r"\b(a|an|the)\s+([.,;:])", r"\2", s, flags=re.I)
    s = re.sub(r"\b(is|are|was|were)\s+(and|or)\b", r"\1", s, flags=re.I)
    # Copula stranded before a comma: "The cardiac silhouette is, top-normal ..."
    # left behind when "stable"/"unchanged" was removed. Hit 2.79% of rows.
    s = re.sub(r"\b(is|are|was|were)\s*,", ",", s, flags=re.I)
    # Quantifier stranded in front of an article after a change-noun was removed:
    # "There is some increase in the size of X" -> "There is some the X" -> "the X"
    s = re.sub(r"\b(some|any|further|overall)\s+(the|a|an)\b", r"\2", s, flags=re.I)
    # Belt-and-braces guard: a lone LOWERCASE consonant is debris from a partial-word
    # match. Case-sensitive on purpose — uppercase singles are real radiology terms
    # ("S-shaped scoliosis", "Kerley B lines", "T-spine"). 'a','i','x' excluded.
    s = re.sub(r"\s+[b-hj-wyz](?=\s*[.,;:])", "", s)
    s = re.sub(r"\s+[b-hj-wyz](?=\s)", "", s)
    # Adverb stranded mid-sentence before a comma when its verb was removed:
    # "Generalized edema slightly, particularly in ..."
    s = re.sub(r"\b(slightly|significantly|essentially|markedly|minimally)\s*,", ",", s, flags=re.I)
    # Preposition stranded before a comma when its object was removed:
    # "the PICC line are in stable position, ..." -> "... are in, ..." -> "... are, ..."
    # Must run BEFORE the copula rule so "are in," collapses fully to "are,"->",".
    s = re.sub(r"\b(in|on|at|of|to|with|from|by)\s*,", ",", s, flags=re.I)
    s = re.sub(r"\b(is|are|was|were)\s*,", ",", s, flags=re.I)
    # Preposition orphaned by "___" removal: "x-ray of ___ where" -> "x-ray of where"
    s = re.sub(r"\b(of|on|in|at|from)\s+(where|when|which)\b", r"\2", s, flags=re.I)
    # Article + participle left when the temporal adverb was removed:
    # "a newly appeared small effusion" -> "a appeared small effusion" -> "a small effusion"
    s = re.sub(r"\b(a|an)\s+(appeared|developed|occurred)\s+(?=\w)", r"\1 ", s, flags=re.I)
    # Copula stranded before a terminal/med-sentence orphan noun left by a strip
    s = re.sub(r"\b(is|are|was|were)\s+(location|studies|exams?|configuration|alignment)\b",
               "", s, flags=re.I)
    s = _punct_fix(s)
    # Strip dangling function words before terminal punctuation, repeatedly
    # ("devices are and." -> "devices are." -> "devices.")
    for _ in range(5):
        s2 = DANGLING.sub(".", s)
        s2 = LEADING.sub("", s2)
        s2 = _punct_fix(s2)
        if s2 == s:
            break
        s = s2
    return s.strip()


def clean_sentence(sent):
    # Returns cleaned sentence, or "" if it should be dropped.
    s = sent.strip()
    if not s:
        return ""

    # E · communication / recommendation -> drop
    if HARD_DROP.search(s):
        return ""

    # A · strip leading comparison clause
    for pat in LEAD_CLAUSE:
        s2 = pat.sub("", s)
        if s2 != s:
            s = s2
            break

    # C · remove de-identification placeholders
    s = re.sub(r"_{2,}", " ", s)

    # B0 · rewrite "re-" prefixed change nouns before stripping
    for pat, repl in PHRASE_REWRITE:
        s = pat.sub(repl, s)

    # B · strip temporal / comparative modifiers.
    # Whitespace MUST be re-collapsed after every substitution: each sub leaves a
    # space behind, so "has been interval decrease" becomes "has been  decrease"
    # (two spaces) and any later multi-token pattern expecting a single space
    # silently stops matching. This is a whole class of silent failures.
    for pat in PHRASE_STRIP_RE:
        s = pat.sub(" ", s)
        s = _WS.sub(" ", s)

    s = _tidy(s)

    # Residual explicit prior reference -> drop the sentence
    if RESIDUAL_PRIOR.search(s):
        return ""

    # D · content gate — must still assert something visible in THIS image
    if not CONTENT_RE.search(s):
        return ""

    # Empty after cleaning. NOTE: do NOT impose a >=3 word minimum here — that
    # deletes legitimate two-word findings like "Moderate cardiomegaly." The
    # content gate above is what filters meaningless fragments.
    if not s.split():
        return ""

    # Normalise terminal punctuation. Strip any trailing ; , : FIRST, otherwise
    # "possible pneumothorax;" becomes "possible pneumothorax;." (hit 529 rows).
    s = re.sub(r"[;,:\-\s]+$", "", s)
    if not s:
        return ""
    if s[-1] not in ".!?":
        s = s + "."
    s = s[0].upper() + s[1:] if len(s) > 1 else s
    # Capitalise after a numbered-list marker: "1. moderate edema." -> "1. Moderate edema."
    s = re.sub(r"^(\d+[.)]\s*)([a-z])", lambda m: m.group(1) + m.group(2).upper(), s)
    return s


# Section headers that leaked INTO the section text during the original MIMIC
# extraction. 32 rows carried a literal "IMPRESSION:" inside findings_text, which
# produced targets reading "FINDINGS: IMPRESSION: ..." — a broken section format
# the decoder would learn to imitate. Strip any embedded header before building.
EMBEDDED_HEADER = re.compile(
    r"\b(FINDINGS|IMPRESSION|IMPRESSIONS|COMPARISON|COMPARISONS|TECHNIQUE|INDICATION"
    r"|HISTORY|EXAMINATION|EXAM|CONCLUSION|REPORT|NOTIFICATION|RECOMMENDATION)\s*:\s*",
    re.I)


def clean_section(text):
    if text is None:
        return ""
    t = str(text).strip()
    if not t or t.lower() in ("nan", "none"):
        return ""
    t = EMBEDDED_HEADER.sub(" ", t)
    kept = [clean_sentence(s) for s in split_sentences(t)]
    kept = [k for k in kept if k]
    return " ".join(kept)


print("Cleaning engine defined.")
print(f"  {len(PHRASE_STRIP_RE)} temporal-modifier patterns")
print(f"  {len(CONTENT_TERMS)} clinical content terms in the safety gate")

---
# 5 · Self-test the engine before touching the corpus

Every case below is a real pattern from your data. **All must pass** — the notebook
raises if any fail, so you never clean 46k reports with a broken rule.

In [ ]:
# ── Unit tests ───────────────────────────────────────────────────────────────
TESTS = [
    # (input, must_contain, must_NOT_contain)
    ("As compared to the previous radiograph, there is no relevant change.",
     [], ["previous", "compared", "change"]),
    ("As compared to the previous radiograph, the pleural effusion has increased.",
     ["effusion"], ["previous", "compared", "increased"]),
    ("Unchanged moderate cardiomegaly with bilateral pleural effusions.",
     ["cardiomegaly", "effusion"], ["Unchanged"]),
    ("Moderate cardiomegaly persists.",
     ["cardiomegaly"], ["persists"]),
    ("The monitoring and support devices are constant.",
     ["devices"], ["constant"]),
    ("Again seen is a small right pleural effusion.",
     ["effusion"], ["Again"]),
    ("These findings were reported to Dr. ___ by telephone at 2:30pm.",
     [], ["telephone", "reported"]),
    ("In comparison with the study of ___, there is little overall change.",
     [], ["comparison", "change"]),
    ("No new parenchymal opacities.",
     ["opacit"], ["new"]),
    ("The lungs are clear without focal consolidation.",
     ["lungs", "clear", "consolidation"], []),
    ("Mild pulmonary edema, unchanged from the prior study.",
     ["edema"], ["unchanged", "prior"]),
    ("Interval removal of the right PICC line.",
     ["picc"], ["Interval"]),
    ("Low lung volumes.", ["lung", "volume"], []),
    ("Stable position of the endotracheal tube.",
     ["endotracheal", "tube"], ["Stable"]),
    ("Repeat radiograph after treatment is recommended.",
     [], ["recommend"]),
    # ── regression tests for bugs found during validation ───────────────────
    # BUG 1: a >=3-word minimum was deleting real two-word findings.
    ("Moderate cardiomegaly.", ["cardiomegaly"], []),
    ("Mild pulmonary edema.", ["edema"], []),
    # BUG 2: `?\b` after an optional group failed before punctuation,
    #        leaking 19% of stability claims through.
    ("Mild cardiac enlargement is unchanged.", ["enlargement"], ["unchanged"]),
    ("Mild cardiomegaly unchanged.", ["cardiomegaly"], ["unchanged"]),
    ("Mediastinal contour including moderate cardiomegaly is stable.",
     ["cardiomegaly"], ["stable"]),
    # BUG 3: dangling function words left ungrammatical targets.
    ("Midline sternotomy wires are again noted.", ["sternotomy"], [" are.", "again"]),
    ("Moderately severe pulmonary edema has worsened since ___.",
     ["edema"], ["worsened", "since"]),
    ("Mediastinal contours are unremarkable and stable.",
     ["mediastinal"], ["stable", " and."]),
    # BUG 4: prior-study reference must not survive in any form.
    ("Cardiomediastinal contours are stable relative to prior exam.",
     ["cardiomediastinal"], ["stable", "prior"]),
    # ── v2 regression tests: bugs found by auditing the v1 output corpus ─────
    # BUG 5: `\bremains?(...)?` matched "remain" inside "remainder" -> "der".
    ("The remainder of the lungs is clear.", ["remainder", "lungs", "clear"], [" der "]),
    ("The remainder of the chest is unremarkable.", ["remainder"], [" der "]),
    # BUG 6: plural prior-nouns escaped ("prior exams", "prior radiographs").
    ("Right apical scarring is unchanged from multiple prior exams.",
     ["scarring"], ["prior", "unchanged"]),
    ("Mild opacity is stable across multiple prior radiographs.",
     ["opacity"], ["prior", "stable"]),
    ("There is minimal right apical pleural thickening as seen on prior radiographs.",
     ["thickening"], ["prior"]),
    # BUG 7: trailing ";" then an appended "." produced ";."
    ("Medial lucency is equivocal for possible pneumothorax; no prior imaging.",
     ["pneumothorax"], [";.", "prior"]),
    # BUG 8: stripping "no significant change" left the stub "There is in ...".
    ("There is no significant change in the extent of the known right pneumothorax.",
     ["pneumothorax"], ["change", " is in "]),
    # ── v3 regression tests: comparative CHANGE language the v2 detector missed ──
    # These imply a previous image without ever using the word "prior".
    ("Increasing opacification of the right hemithorax likely representing "
     "reaccumulation of pleural fluid.",
     ["opacification", "pleural fluid"], ["increasing", "reaccumulation"]),
    ("There has been interval decrease in lung volumes.",
     ["lung volumes"], ["decrease", "interval", "has been"]),
    ("There is some increase in the size of the left pleural effusion.",
     ["effusion"], ["increase"]),
    ("Worsening pulmonary vascular congestion.",
     ["congestion"], ["worsening"]),
    ("Right lung base opacity is similar.", ["opacity"], ["similar"]),
    ("Mediastinal surgical clips are unchanged in location.",
     ["clips"], ["unchanged", "location"]),
    ("The cardiac silhouette is stable, top-normal to mildly enlarged.",
     ["cardiac silhouette", "enlarged"], ["stable", " is,"]),
    ("Aortic aneurysm is likely secondary to the previously imaged arch aneurysm.",
     ["aneurysm"], ["previously", "the imaged"]),
    ("Fibrotic changes at the right apex similar to prior studies.",
     ["apex"], ["prior", "studies", "similar"]),
    # ── v4 regression tests: defects found by auditing the v3 output corpus ──
    # BUG 9: alternation without trailing \b matched "increase" inside "increased"
    #        and left the orphan letter "d". 2.5% of rows.
    ("The pleural effusion has increased.", ["effusion"], ["increased", " d"]),
    ("The right pleural effusion has decreased.", ["effusion"], ["decreased"]),
    ("Left basal opacity has resolved.", ["opacity"], ["resolved"]),
    # BUG 10: bare change nouns survived.
    ("1. Improvement in left pleural effusion.", ["effusion"], ["improvement"]),
    ("Worsening of the right basilar atelectasis.", ["atelectasis"], ["worsening"]),
    # BUG 11: adverbs stranded when their verb was removed.
    ("The left pleural effusion has improved significantly.",
     ["effusion"], ["improved", "significantly"]),
    ("Prior opacity at the left base is essentially resolved.",
     ["opacity"], ["prior", "essentially", "resolved"]),
    # BUG 12: bare "prior" used adjectivally still asserts a previous image.
    ("Prior granuloma in the right upper lobe.", ["granuloma"], ["prior"]),
    # BUG 13: lowercase after a numbered-list marker.
    ("1. Increased mild pulmonary edema.", ["1. Mild"], ["increased"]),
    # ── v5 regression tests: root causes found by auditing the v4 corpus ─────
    # BUG 14: "again" without \b matched inside "against" -> orphan "st".
    ("The catheter lies against the chest wall.", ["against", "chest wall"], [" st "]),
    ("Left lower lobe atelectasis is decreased and results in mediastinal shift.",
     ["atelectasis"], ["decreased", " d "]),
    # BUG 15: DANGLING required trailing punctuation, so a stripped sentence with
    #         no period kept its stranded copula.
    ("Frontal and lateral views of the chest are unchanged.",
     ["chest"], ["unchanged", " are."]),
    ("Small left pleural effusion is stable.", ["effusion"], ["stable", " is."]),
    ("Bilateral pleural effusions are unchanged.", ["effusions"], ["unchanged", " are."]),
    ("Moderate cardiomegaly has been stable dating back to at least 2019.",
     ["cardiomegaly"], ["stable", " s "]),
    # ── v6 regression tests: grammar debris found in the v5 output corpus ────
    # BUG 16: preposition stranded before a comma (1.08% of rows).
    ("The tracheostomy tube and the right PICC line are in stable position, "
     "the PICC line needs to be pulled back.",
     ["PICC"], ["stable", " in,", " are,"]),
    # BUG 17: preposition orphaned by "___" removal.
    ("As shown on the chest x-ray of ___ where there were hyperinflated lungs.",
     ["hyperinflated"], ["of where"]),
    # BUG 18: article + participle left when the temporal adverb was removed.
    ("There is a newly appeared small left pleural effusion.",
     ["a small left pleural effusion"], ["newly", "a appeared"]),
    # BUG 19: two-digit / paren list markers were not capitalised.
    ("10) increased bibasilar atelectasis.", ["10) Bibasilar"], ["increased"]),
    # BUG 20: section headers leaked into the source section text, producing
    #         targets that read "FINDINGS: IMPRESSION: ...". 32 rows.
    ("IMPRESSION: The lungs are clear.", ["lungs are clear"], ["impression:"]),
    ("COMPARISON: none. The heart size is normal.", ["heart size"], ["comparison:"]),
    # ── v8 regression tests: FINDING-LOSS bugs found by the acceptance audit ──
    # BUG 21: HARD_DROP on "comparison with/to" deleted whole sentences that
    #         carried the only mention of a pathology.
    ("In comparison with the study of ___, there has been the development of a "
     "moderate post-procedure pneumothorax.",
     ["pneumothorax"], ["comparison", "___"]),
    ("In comparison with the study of ___, there again is a small right apical "
     "pneumothorax.", ["pneumothorax"], ["comparison", "again"]),
    ("In comparison with the prior study, there is little change in the degree of "
     "left apical pneumothorax with chest tube in place.",
     ["pneumothorax", "chest tube"], ["comparison", "prior", "change"]),
    # Genuinely contentless comparison sentences must STILL be dropped.
    ("Comparison is made with the prior study.", [], ["comparison", "prior"]),
    ("Comparison to ___.", [], ["comparison"]),
    # BUG 22: a clock time merged two sentences, so one sentence's deletion took
    #         the other's findings with it.
    ("Comparison to ___, 13:45. The known right pneumothorax is not substantially "
     "changed.", ["pneumothorax"], ["comparison"]),
]

print("=" * 78)
print("  CLEANING ENGINE · SELF-TEST")
print("=" * 78)
fails = []
for i, (src, must, mustnot) in enumerate(TESTS, 1):
    out = clean_section(src)
    lo = out.lower()
    ok = True
    why = []
    for m in must:
        if m.lower() not in lo:
            ok = False; why.append(f"lost '{m}'")
    for m in mustnot:
        if m.lower() in lo:
            ok = False; why.append(f"kept '{m}'")
    # ── UNIVERSAL INVARIANTS ────────────────────────────────────────────────
    # Applied to EVERY test regardless of its own assertions. Per-test checks are
    # too weak on their own: test [02] once passed while emitting "The pleural
    # effusion d." because it only asserted "effusion" present / "increased" absent.
    # Debris from partial-word matches is a silent, corpus-wide failure mode.
    if out:
        # Case-sensitive: uppercase singles are legitimate ("Kerley B lines").
        if re.search(r"\s[b-hj-wyz][\s.,;:]", " " + out + " "):
            ok = False; why.append("DEBRIS: orphan single letter (partial-word match)")
        if "  " in out:
            ok = False; why.append("DEBRIS: double space")
        if re.search(r"[;,:]\s*\.", out):
            ok = False; why.append("DEBRIS: punctuation artefact")
        if re.search(r"\b(is|are|was|were)\s*[.,]", out, re.I):
            ok = False; why.append("DEBRIS: stranded copula")
    flag = "✅" if ok else "❌"
    print(f"  {flag} [{i:02d}] {src[:62]}")
    print(f"        -> {out if out else '(dropped)'}")
    if not ok:
        print(f"        !! {'; '.join(why)}")
        fails.append((i, src, out, why))

print("=" * 78)
if fails:
    print(f"  ❌ {len(fails)} / {len(TESTS)} tests FAILED — do not proceed.")
    raise AssertionError(f"{len(fails)} cleaning-engine self-tests failed: {fails}")
print(f"  ✅ ALL {len(TESTS)} TESTS PASSED — engine is safe to run on the corpus.")
print("=" * 78)

---
# 6 · Apply cleaning to all splits

`FINDINGS:` and `IMPRESSION:` section tokens are added in the **correct radiological
order** (findings before impression). Your current pipeline concatenates them
inverted, which teaches the model to emit the conclusion first — exactly where mode
collapse bites hardest.

In [ ]:
# ── Target construction ──────────────────────────────────────────────────────
MIN_WORDS = 8
USE_SECTION_TOKENS = True   # set False for plain prose targets


def build_clean_target(imp_clean, fin_clean):
    imp, fin = imp_clean.strip(), fin_clean.strip()
    if USE_SECTION_TOKENS:
        parts = []
        if fin:
            parts.append("FINDINGS: " + fin)
        if imp:
            parts.append("IMPRESSION: " + imp)
        out = " ".join(parts).strip()
        # Guard: never emit a section header with no content behind it.
        out = re.sub(r"\s*(FINDINGS|IMPRESSION):\s*$", "", out).strip()
        return out
    return (fin + " " + imp).strip()


CLEAN = {}
stage_stats = {}

for s in SPLITS:
    df = DF[s].copy()
    print(f"\n{'=' * 78}\n  CLEANING · {s.upper()}  ({len(df):,} rows)\n{'=' * 78}")

    tqdm.pandas(desc=f"  {s} impression")
    df["impression_clean"] = df["impression_text"].progress_apply(clean_section)
    tqdm.pandas(desc=f"  {s} findings  ")
    df["findings_clean"] = df["findings_text"].progress_apply(clean_section)

    df["report_clean"] = [build_clean_target(a, b) for a, b in
                          zip(df["impression_clean"], df["findings_clean"])]
    df["_orig_target"] = df.apply(build_original_target, axis=1)

    n_before = len(df)
    wc = df["report_clean"].str.split().str.len().fillna(0)
    keep = wc >= MIN_WORDS
    n_dropped = int((~keep).sum())
    df = df[keep].reset_index(drop=True)

    CLEAN[s] = df
    stage_stats[s] = {
        "rows_before": n_before,
        "rows_after": int(len(df)),
        "rows_dropped_short": n_dropped,
        "retention_pct": round(len(df) / n_before * 100, 2),
    }
    print(f"\n  rows: {n_before:,} -> {len(df):,}  "
          f"(dropped {n_dropped:,} under {MIN_WORDS} words, "
          f"retention {len(df)/n_before*100:.2f}%)")

In [ ]:
# ── Length & label-integrity comparison ─────────────────────────────────────
print("=" * 78)
print("  BEFORE / AFTER")
print("=" * 78)
print(f"  {'split':<8}{'rows':>18}{'orig words':>14}{'clean words':>14}{'retained':>11}")
print(f"  {'-'*8}{'-'*18}{'-'*14}{'-'*14}{'-'*11}")
for s in SPLITS:
    d = CLEAN[s]
    ow = d["_orig_target"].str.split().str.len().mean()
    cw = d["report_clean"].str.split().str.len().mean()
    st = stage_stats[s]
    print(f"  {s:<8}{st['rows_before']:>8,} -> {st['rows_after']:>6,}"
          f"{ow:>14.1f}{cw:>14.1f}{st['retention_pct']:>10.2f}%")

# Label prevalence must not have shifted materially
print(f"\n  LABEL PREVALENCE DRIFT (train, before -> after):")
print(f"  {'Label':<20}{'before%':>10}{'after%':>10}{'drift':>9}")
drift_ok = True
label_drift = {}
for c in LABEL_COLS_FULL:
    if c not in DF["train"].columns:
        continue
    b = (DF["train"][c] == 1).mean() * 100
    a = (CLEAN["train"][c] == 1).mean() * 100
    d = a - b
    label_drift[c] = {"before": round(b, 2), "after": round(a, 2), "drift": round(d, 2)}
    flag = "" if abs(d) < 2.0 else "  ⚠️"
    if abs(d) >= 2.0:
        drift_ok = False
    print(f"  {c:<20}{b:>10.2f}{a:>10.2f}{d:>+9.2f}{flag}")
print(f"\n  {'✅ No material label drift.' if drift_ok else '⚠️  Some labels drifted >2pp — review which rows were dropped.'}")

---
# 7 · QA GATE — did we actually remove the hallucination source?

The same detector from Section 3, re-run on the cleaned corpus. **Target: < 2%.**

In [ ]:
print("=" * 78)
print("  POST-CLEAN QA GATE")
print("=" * 78)

ALL_CLEAN = pd.concat([CLEAN[s]["report_clean"] for s in SPLITS], ignore_index=True)
post_audit = audit_prior_language(ALL_CLEAN, "ALL SPLITS · cleaned targets")

print("\n" + "=" * 78)
print("  BEFORE -> AFTER")
print("=" * 78)
print(f"  {'pattern':<24}{'before':>10}{'after':>10}{'reduction':>12}")
print(f"  {'-'*24}{'-'*10}{'-'*10}{'-'*12}")
for k in list(DETECT.keys()) + ["ANY"]:
    b = pre_audit[k]["pct"]; a = post_audit[k]["pct"]
    red = (1 - a / b) * 100 if b > 0 else 0.0
    print(f"  {k:<24}{b:>9.2f}%{a:>9.2f}%{red:>11.1f}%")

# Sentence level
samp2 = ALL_CLEAN.sample(min(6000, len(ALL_CLEAN)), random_state=RNG_SEED)
tot2 = bad2 = 0
for t in samp2:
    for s_ in split_sentences(t):
        if len(s_) < 5:
            continue
        tot2 += 1
        if DETECT_ANY.search(s_):
            bad2 += 1
sent_rate_post = bad2 / max(tot2, 1) * 100
print(f"\n  SENTENCE-LEVEL: {sent_rate:.1f}% -> {sent_rate_post:.2f}%")

# ── Gates ────────────────────────────────────────────────────────────────────
residual = post_audit["ANY"]["pct"]
retention = np.mean([stage_stats[s]["retention_pct"] for s in SPLITS])
empty = int((ALL_CLEAN.str.strip().str.len() == 0).sum())

print("\n" + "=" * 78)
print("  SUCCESS CRITERIA")
print("=" * 78)
# Grammar/corruption artefacts introduced BY the cleaning itself. The prior-reference
# gates can pass while the engine is quietly producing malformed text, so these are
# checked separately. Each was a real defect found by auditing an earlier output corpus.
ARTEFACTS = {
    "orphan lowercase letter": r"\s[b-hj-wyz][\s.,;:]",
    "truncated word (remainder->der)": r"\bder\b",
    "dangling adverb": r"\b(significantly|essentially|slightly|markedly)\s*[.,]",
    "';.' double punctuation": r";\s*\.",
    "stranded copula 'is ,'": r"\b(is|are|was|were)\s*,",
    "orphan noun ('are location')": r"\b(is|are)\s+(location|studies|exams?|configuration|alignment)\b",
    "quantifier + article ('some the')": r"\b(some|any)\s+(the|a|an)\b",
    "double space": r"  ",
    # NB: "the imaged" is NOT listed — it is legitimate radiology English
    # ("the imaged upper abdomen") and occurs in the ORIGINALS at the same 0.18%.
}
art_total = 0
print("\n  CLEANING ARTEFACTS (defects the cleaner itself could introduce):")
for aname, apat in ARTEFACTS.items():
    ahits = int(ALL_CLEAN.str.contains(apat, regex=True, na=False).sum())
    art_total += ahits
    print(f"    {aname:<34}{ahits:>7,}  {ahits/len(ALL_CLEAN)*100:>6.3f}%")
art_pct = art_total / len(ALL_CLEAN) * 100

gates = [
    ("Residual prior-reference rate < 0.5%", residual, residual < 0.5, f"{residual:.2f}%"),
    ("Sentence-level residual < 0.5%", sent_rate_post, sent_rate_post < 0.5, f"{sent_rate_post:.2f}%"),
    ("Cleaning artefacts < 1%", art_pct, art_pct < 1.0, f"{art_pct:.2f}%"),
    ("Corpus retention > 92%", retention, retention > 92.0, f"{retention:.2f}%"),
    ("Zero empty targets", empty, empty == 0, str(empty)),
]
all_pass = True
for name, _, ok, val in gates:
    print(f"  {'✅' if ok else '❌'} {name:<42} {val:>10}")
    if not ok:
        all_pass = False
print("=" * 78)
print(f"  {'✅ STAGE 1 PASSED' if all_pass else '⚠️  REVIEW NEEDED — see failures above'}")
print("=" * 78)
if not all_pass:
    print("\n  Not fatal — inspect the examples in the next cell and tune")
    print("  PHRASE_STRIP / CONTENT_TERMS, then re-run from Section 4.")

---
# 8 · Eyeball the result

Automated gates are necessary but not sufficient. **Read these.** You are the
clinical judge of whether the cleaned targets still say something true.

In [ ]:
ex_lines = []
rng = np.random.RandomState(RNG_SEED)
pool = CLEAN["train"]
# Prefer examples that originally contained prior-language (the hard cases)
had_prior = pool["_orig_target"].str.lower().str.contains(DETECT_ANY.pattern, regex=True, na=False)
idx = list(rng.choice(pool.index[had_prior], size=min(30, int(had_prior.sum())), replace=False))
idx += list(rng.choice(pool.index[~had_prior], size=min(10, int((~had_prior).sum())), replace=False))

for n, i in enumerate(idx, 1):
    r = pool.loc[i]
    before = " ".join(str(r["_orig_target"]).split())
    after = " ".join(str(r["report_clean"]).split())
    block = (f"--- [{n:02d}] dicom={r['dicom_id']}\n"
             f"  BEFORE: {before}\n"
             f"  AFTER : {after}\n")
    ex_lines.append(block)
    if n <= 12:
        print(block)

print(f"\n({len(idx)} examples written to stage1_examples.txt)")

---
# 9 · Constant-baseline control (recomputed on the CLEANED corpus)

On your **original** test set a single fixed string — ignoring the X-ray entirely —
scored **ROUGE-L 0.2481**, versus your trained model's **0.2739**. The entire vision
pipeline was worth **+0.026**.

Cleaning changes the reference distribution, so the baseline must be recomputed.
**This is the number to beat in Stage 4.** Any ROUGE-L you report later is
meaningless without it.

In [ ]:
from rouge_score import rouge_scorer
_sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

N_CAND = 150     # candidate pool from most-frequent cleaned training targets
N_VAL  = 300     # val sample used to pick the best constant
test_refs = CLEAN["test"]["report_clean"].tolist()

cands = CLEAN["train"]["report_clean"].value_counts().head(N_CAND).index.tolist()
val_refs = CLEAN["val"]["report_clean"].sample(
    min(N_VAL, len(CLEAN["val"])), random_state=RNG_SEED).tolist()

print(f"Searching {len(cands)} candidates against {len(val_refs)} val refs...")
best, best_score = None, -1.0
for c in tqdm(cands, desc="  candidate search"):
    m = np.mean([_sc.score(r, c)["rougeL"].fmeasure for r in val_refs])
    if m > best_score:
        best, best_score = c, m

print(f"\n  Best constant (val ROUGE-L = {best_score:.4f}):")
print(f'  "{best[:200]}"')

print(f"\nScoring on full cleaned test set (n={len(test_refs):,})...")
r1 = r2 = rl = 0.0
for r in tqdm(test_refs, desc="  test scoring"):
    sc_ = _sc.score(r, best)
    r1 += sc_["rouge1"].fmeasure; r2 += sc_["rouge2"].fmeasure; rl += sc_["rougeL"].fmeasure
n = len(test_refs)
CONST_R1, CONST_R2, CONST_RL = r1/n, r2/n, rl/n

print("\n" + "=" * 78)
print("  CONSTANT-OUTPUT BASELINE ON CLEANED TEST SET")
print("=" * 78)
print(f"  ROUGE-1: {CONST_R1:.4f}   ROUGE-2: {CONST_R2:.4f}   ROUGE-L: {CONST_RL:.4f}")
print()
print(f"  ⚠️  ANY MODEL YOU TRAIN MUST BEAT ROUGE-L {CONST_RL:.4f} TO HAVE LEARNED ANYTHING.")
print(f"      Report 'margin over constant baseline', not raw ROUGE-L.")
print(f"      Reference point: on the ORIGINAL corpus this was 0.2481 vs model 0.2739 (+0.026).")
print("=" * 78)

---
# 10 · Save artifacts

In [ ]:
EXPORT_COLS = [c for c in [
    "dicom_id", "subject_id", "study_id", "image_rel_path", "ViewPosition",
    *LABEL_COLS_FULL,
    "impression_clean", "findings_clean", "report_clean",
    "impression_text", "findings_text",
] if c in CLEAN["train"].columns]

print("=" * 78)
print("  SAVING")
print("=" * 78)
for s in SPLITS:
    p = CLEAN_DIR / f"clean_{s}.csv"
    CLEAN[s][EXPORT_COLS].to_csv(p, index=False)
    print(f"  ✅ {p}   ({len(CLEAN[s]):,} rows, {p.stat().st_size/1e6:.1f} MB)")

(REPORT_DIR / "stage1_examples.txt").write_text("\n".join(ex_lines), encoding="utf-8")
print(f"  ✅ {REPORT_DIR / 'stage1_examples.txt'}")

audit_out = OrderedDict([
    ("stage", "1 · report target cleaning"),
    ("timestamp", datetime.now().isoformat()),
    ("config", {"min_words": MIN_WORDS, "section_tokens": USE_SECTION_TOKENS,
                "seed": RNG_SEED, "n_phrase_patterns": len(PHRASE_STRIP_RE),
                "n_content_terms": len(CONTENT_TERMS)}),
    ("integrity", integrity),
    ("pre_clean", {"pattern_rates": pre_audit, **pre_stats}),
    ("post_clean", {"pattern_rates": post_audit,
                    "sentence_level_ungroundable_pct": round(sent_rate_post, 2)}),
    ("split_stats", stage_stats),
    ("label_drift_train", label_drift),
    ("constant_baseline_cleaned_test",
     {"rouge1": round(CONST_R1, 4), "rouge2": round(CONST_R2, 4),
      "rougeL": round(CONST_RL, 4), "string": best}),
    ("constant_baseline_original_test", {"rougeL": 0.2481, "model_rougeL": 0.2739}),
    ("gates_passed", bool(all_pass)),
])
p = REPORT_DIR / "stage1_audit.json"
p.write_text(json.dumps(audit_out, indent=2, default=str), encoding="utf-8")
print(f"  ✅ {p}")
print("=" * 78)

---
# 11 · Stage 1 summary

In [ ]:
print("=" * 78)
print("  COMPONENT_01 · STAGE 1 COMPLETE")
print("=" * 78)
print(f"""
  HALLUCINATION SOURCE
    prior-reference reports    {pre_audit['ANY']['pct']:>6.2f}%  ->  {post_audit['ANY']['pct']:>5.2f}%
    un-groundable sentences    {sent_rate:>6.2f}%  ->  {sent_rate_post:>5.2f}%

  CORPUS
    train   {stage_stats['train']['rows_before']:>7,} -> {stage_stats['train']['rows_after']:>7,}   ({stage_stats['train']['retention_pct']:.2f}% retained)
    val     {stage_stats['val']['rows_before']:>7,} -> {stage_stats['val']['rows_after']:>7,}   ({stage_stats['val']['retention_pct']:.2f}% retained)
    test    {stage_stats['test']['rows_before']:>7,} -> {stage_stats['test']['rows_after']:>7,}   ({stage_stats['test']['retention_pct']:.2f}% retained)

  TARGET FORMAT
    FINDINGS: ... IMPRESSION: ...   (correct radiological order — was inverted)

  CONTROL NUMBER FOR STAGE 4
    constant-baseline ROUGE-L = {CONST_RL:.4f}
    -> your model must beat this to have learned anything from the image

  OUTPUTS
    {CLEAN_DIR}/clean_train.csv
    {CLEAN_DIR}/clean_val.csv
    {CLEAN_DIR}/clean_test.csv
    {REPORT_DIR}/stage1_audit.json
    {REPORT_DIR}/stage1_examples.txt
""")
print("=" * 78)
print("  Compute units consumed: 0 (CPU runtime)")
print("=" * 78)

---
# 12 · GPU & Colab purchasing guide (for Stage 4)

**You do not need any of this for Stage 1.** Read it before Stage 4.

## Which plan to buy

**Colab Pro — $9.99/month = 100 compute units.**

Buy at `colab.research.google.com` → *Runtime → Change runtime type* → the upgrade
prompt, or directly via **Colab Pro** in the top-right menu. Pay-as-you-go ($9.99
per 100 units, no subscription) is also fine if you only need one training run.

## Which GPU to select

`Runtime → Change runtime type → Hardware accelerator`

| GPU | Compute units/hr | Hours from 100 CU | Verdict |
|---|---|---|---|
| **CPU** | **0** | ∞ | **Stages 1–3 — use this** |
| T4 | ~1.76 | ~57 h | Cheap but ~3× slower, 16 GB, no bf16 |
| **L4** | ~4.8 | ~20 h | ⭐ **Use for Stage 4** |
| A100 40GB | ~15 | ~7 h | Burns too fast for your budget |

**Pick L4.** A100 leaves you under 7 hours — not enough for a 6–10 h training run
plus retries. T4 is cheaper per hour but ~3× slower, so it costs roughly the same
per unit of work while tripling your wall-clock and therefore your disconnect risk.

Your Stage 4 + Stage 5 need is **14–22 GPU-hours**, so 100 CU covers Stage 4
comfortably. Budget **~$20** if you also want Stage 5.

*Rates change — check the live figure in the Colab UI before a long run.*

## How to not burn units

1. **Units burn while the GPU is *connected*, not while training.** An idle notebook
   with a GPU attached is draining you. Always `Runtime → Disconnect and delete runtime`.
2. **Stages 1–3 cost zero units.** Run them on CPU. This alone saves a third of your budget.
3. **Never debug on GPU.** Get it running on CPU with 10 samples, batch size 2, first.
4. **Checkpoint to Drive every epoch.** A disconnect at hour 8 with no checkpoint is
   ~40 units gone.
5. **Tar your images before uploading.** 46k loose PNGs on Drive will leave the GPU
   idle waiting on I/O — paying GPU rates to read files.

---

## Next: Stage 2 & 3 (also CPU, also free)

- **Stage 2** — transform fixes: per-image intensity normalisation or CLAHE
  (measured per-image mean drifts 97.8 → 126.2, uncorrected); delete `ColorJitter`
  and `RandomAutocontrast` (for CXR, intensity *is* the signal); rotation 10° → 5°;
  fix the train/inference resize skew.
- **Stage 3** — swap to the official CheXpert labels already on disk.

**Then Stage 4** — the architecture rebuild, which is what actually moves ROUGE-L:
stop bypassing BART's encoder (`train_report_generator.py:172`), add 2D positional
embeddings to the 144 visual tokens, replace the single `Linear(1024→768)` with a
cross-attention resampler, and swap in BioBART.

---

### ⚠️ Do not train on the old CSVs

From here on, point every training script at `data/stage1_clean/clean_*.csv` and the
`report_clean` column. Evaluating a cleaned model against uncleaned references — or
vice versa — produces meaningless numbers.